**10 언어 모델을 위한 신경망**
====

**10-1 어텐션 메커니즘과 트랜스포머**
----

**기계 번역 machine translation**    
- 번역할 문장이 길어질수록 기존 RNN 기반 모델은 번역의 품질 유지 어려움     

**sequence-to-sequence 구조**    
- 텍스트를 입력받아 텍스트를 출력하는 작업     
ex) 기계 번역, 문서 요약     

- 보통 **인코더-디코더 구조** 사용         
    각각 순환 신경망을 적용 (인코더 RNN, 디코더 RNN 존재)       
    인코더, 디코더 모두 한 토큰씩 처리     
    1. 디코더가 1개 토큰 생성      
    2. 인코더의 마지막 은닉 상태와 디코더의 은닉 상태 활용해서 다음 디코더의 토큰 생성     
    -> **자기회귀 모델 autoregressive model**       

**어텐션 메커니즘 attetion mechanism**      
- RNN 기반 인코더-디코더 모델 성능을 크게 향상시킨 기술     
- 인코더의 모든 타임스텝에서 계산된 은닉 상태를 활용 가능     

- **어텐션 가중치**      
    인코더의 은닉 상태마다 가중치를 다르게 적용하여 더 중요한 정보를 강조     

- **장점** : **긴 텍스트를 처리**할 때 정보 손실을 줄이는 데 효과적

- **단점** : **연산량 증가**      
            인코더가 처리할 수 있는 타임스텝의 최대 개수 정해야 함       
            입력 텍스트의 길이가 제한될 수 있음        
            인코더의 토큰을 여전히 하나씩 처리        

**트랜스포머 Transfomer**      
- 어텐션 메커니즘 활용 (+다양한 기술 조합)      
- 기존 인코더-디코더 구조를 유지하면서 **RNN 제거**한 모델     
- 입력 텍스트를 한 토큰씩 처리할 필요 없이 **한 번에 모두 처리 가능**        

- **작동 방식**      
    - 입력된 텍스트를 한 번에 모두 처리 -> 처리 속도 향상      
    - 타임스텝 개념이 필요 없음        
    - 어텐션 메커니즘으로 인해 사실상 입력 텍스트의 길이에 제한이 있음     

- **전체 구조**      
    **Encoder + Decoder 구조**            

    - **Encoder** : 입력 문장을 보고 의미를 벡터로 압축         
        - Self-Attention
        - Feed-Forward Network         
    
    - **Decoder**     
        - 은닉 벡터를 활용해 각 타임스테베서 출력할 토큰 생성    
        - 이전에 생성된 토큰을 참고하면서 새로운 토큰을 만듦     
        - 자기 회귀 방식      

**Self-Attention**     
- 인코더에 입력되는 토큰만으로 어텐션 가중치를 학습       
- Self-Attention 계산 과정            
    - 입력 텍스트의 각 토큰이 밀집층1에 통과 -> **쿼리 벡터** 생성           
    - 입력 텍스트를 밀집층2에 통과 -> **키 벡터** 생성            
- 어텐션 점수 계산       
    - 쿼리 벡터 * 키 벡터 = 어텐션 점수 -> 어텐션 행렬이 만들어 짐      
    - ex) 입력된 토큰 3개 : 쿼리 벡터 3개, 키 벡터 3개 -> 어텐션 점수 9개        
    - 입력 텍스트를 밀집층3에 통과 -> **값 벡터**      
    - 값 벡터 * 어텐션 점수 -> 최종적인 셀프 어텐션 출력 -> 각 입력 토큰이 다른 토큰들과 얼마나 관련이 있는지 반영한 **은닉벡터**와 같다         
    - 이를 통해 모델은 문맥을 더 정확히 이해하고, 중요한 정보를 효과적으로 강조       

- **멀티 헤드 어텐션**     
    - **어텐션 헤드** : 셀프 어텐션 연산을 수행하는 하나의 단위      
    - 트랜스포머는 여러 개의 어텐션 헤드를 사용      
    - 각 어텐션 헤드의 출력은 하나로 합쳐진 후, 밀집층을 통과하여 어텐션 층의 최종 출력이 된다      

**정규화**     
- DL에서는 여러 개의 층을 거치면서 특성의 스케일이 변할 수 있음       
- 단순한 입력 정규화만으로 충분하지 않음       
    - **배치 정규화**       
        - 주로 합성곱 신경망에 널리 활용, 층과 층 사이에 놓임       
        - 이전 층의 출력을 배치 단위로 평균=0, 분산=1이 되도록 조정 후 다음 층으로 전달      
        - 훈련 속도가 빨라지고, 학습 과정이 안정화, 모델 성능 향상, 많은 신경망에서 널리 사용      
        - 단점 : 텍스트 데이터에 적용하기는 어려움 (샘플마다 길이가 다르기 때문)         
    - **층 정규화**       
        - 각 샘플의 토큰마다 개별적으로 정규화를 수행하는 방식        
        - 샘플마다 길이가 달라도 독립적으로 정규화 가능      
        - 멀티 헤드 어텐션 / 드롭아웃, 층 정규화 (일부 모델에서는 반대 순서)      
        - **잔차 연결 = 스킵 연결**       
            - 멀티 헤드 어텐션 층을 거친 출력에 입력값을 더하는 방식      
            - 신경망의 층을 많이 쌓아도 효과적으로 훈련     

**피드포워드 네트워크**       
- 트랜스포머의 인코더 - 멀티 헤드 어텐션 - 층 정규화 - 밀집층     
- 2개의 밀집층      
    - 첫 번째 밀집층 : ReLU 활성화 함수 사용     
    - 두 번째 밀집층 : 활성화 함수 사용 안함      
    - 드롭아웃 층
- 잔차 연결이 3개의 층을 감싼다     
- 단어 임베딩(입력값)와 은닉 벡터(출력값)은 벡터의 차원이 동일하다     
- 이로 인해 인코더 블록은 여러 개 반복해서 배치 가능     

**토큰 임베딩**       
- 트랜스포머는 모든 토큰을 동시에 처리하는 방식     
- 토큰의 위치를 고려하지 않는다 -> 위치 정보 필요      
**위치 임베딩**      
- **위치 인코딩**      
    - 사인 함수(짝수 번째 원소)와 코사인 함수(홀수 번째 원소)를 사용해 토큰의 위치에 따라 변하는 벡터 생성 -> 단어 임베딩에 더함      
    - 원본 단어 임베딩은 문자의 위치에 따라 달라짐      
    - 토큰의 위치와 임베딩 벡터의 차원에 따라 일정한 값으로 계산 -> 절대 인코딩      

**디코더 블록**       
- **크로스 어텐션**     
    - 인코더가 출력한 임베딩 벡터를 입력으로 받는 멀티 헤드 어텐션 층     
    - 디코더에서 받은 벡터 : 쿼리      
    - 인코더의 출력 : 키, 값      
    - 위치 : 인코더의 멀티 헤드 어텐션 층 / 디코더 블록 / 피드포워드 네트워크       
- 디코더 블록도 여러 개가 반복적으로 쌓여 디코더 모델 구성     
- 인코더 블록의 출력은 모든 디코더 블록에 전달     
- 자기 회귀 모델의 방식에 따라 한 번에 하나의 토큰만 생성     
- **마스킹**       
    - 한 타임스텝에서 어텐션 점수를 계산할 때 현재 토큰까지만 참고하고, 이후의 토큰은 볼 수 없도록 제한         
    - 첫 번째 멀티 헤드 어텐션 층에 적용 (마스크드 멀티 헤드 어텐션층)



